# Customer support ticket router with path history

This LangGraph example routes tickets to billing or technical support using keyword rules. An `operator.add` reducer accumulates the names of visited nodes. No API key is required.

**Prerequisites:** Google Colab with a Python runtime and internet access to install LangGraph. Upload this notebook and choose **Runtime → Run all**.

Routing is case-insensitive. Tickets containing `billing`, `invoice`, `payment`, `refund`, or `charge` go to billing; all others go to technical support. Billing takes precedence when a ticket mentions both kinds of issue.

In [ ]:
%pip install -q langgraph


In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from IPython.display import Markdown, display
from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: Literal['billing', 'technical'] | None
    reply: str
    history: Annotated[list[str], operator.add]


def classify_request(state: TicketState) -> dict:
    keywords = ('billing', 'invoice', 'payment', 'refund', 'charge')
    ticket = state['ticket'].lower()
    category = 'billing' if any(word in ticket for word in keywords) else 'technical'
    return {'category': category, 'history': ['classify_request']}


def select_support_team(state: TicketState) -> Literal['billing', 'technical']:
    if state['category'] == 'billing':
        return 'billing'
    return 'technical'


def handle_billing(state: TicketState) -> dict:
    return {
        'reply': 'Billing support will review your payment or invoice issue.',
        'history': ['handle_billing'],
    }


def handle_technical(state: TicketState) -> dict:
    return {
        'reply': 'Technical support will help troubleshoot your issue.',
        'history': ['handle_technical'],
    }


builder = StateGraph(TicketState)
builder.add_node('classify_request', classify_request)
builder.add_node('handle_billing', handle_billing)
builder.add_node('handle_technical', handle_technical)
builder.add_edge(START, 'classify_request')
builder.add_conditional_edges(
    'classify_request',
    select_support_team,
    {'billing': 'handle_billing', 'technical': 'handle_technical'},
)
builder.add_edge('handle_billing', END)
builder.add_edge('handle_technical', END)
graph = builder.compile()


## Graph diagram
Colab renders the Mermaid diagram below, showing both conditional routes.

In [ ]:
display(Markdown('```mermaid\n' + graph.get_graph().draw_mermaid() + '\n```'))


## Sample tickets and expected output

Each invocation starts with empty history. Each node returns only its own name; the reducer appends that update to previous history. The routing function selects an edge and is not a graph node.

```text
Ticket: I was charged twice for my invoice.
Category: billing
Reply: Billing support will review your payment or invoice issue.
History: classify_request -> handle_billing

Ticket: The app crashes whenever I open settings.
Category: technical
Reply: Technical support will help troubleshoot your issue.
History: classify_request -> handle_technical
```

In [ ]:
sample_tickets = [
    'I was charged twice for my invoice.',
    'The app crashes whenever I open settings.',
]

for ticket in sample_tickets:
    result = graph.invoke({
        'ticket': ticket,
        'category': None,
        'reply': '',
        'history': [],
    })
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}")
    print(f"History: {' -> '.join(result['history'])}")
    print()
